# Session 2 — Lakehouse Monitoring Setup

**Goal:** Create a Lakehouse Monitor on the inference log table.

Lakehouse Monitoring will:
- Compare the current inference distribution vs a baseline (training data)
- Compute drift metrics (PSI) for every feature
- Track model performance metrics over time
- Auto-generate a monitoring dashboard in Unity Catalog

In [0]:
%run ../utils/config

In [0]:
inference_table = f"{catalog}.{schema}.churn_inference_log" # Must be in three level namespace format
print(f"Monitoring table : {inference_table}")
print(f"Baseline table   : {catalog}.`00_shared`.telco_churn")

## Check the Inference Log

Before creating the monitor, verify the inference log has the expected shape.

In [0]:
df_log = spark.table(inference_table)
print(f"Rows: {df_log.count():,}")
print(f"Columns: {df_log.columns}")
display(df_log.limit(3))

## Create the Lakehouse Monitor

This creates a monitor with:
- **InferenceLog** profile type: tracks prediction distribution + model accuracy
- **Baseline**: a view of `workshop.00_shared.telco_churn` with schema-aligned types
- **Granularity**: daily windows
- **Output**: auto-creates `churn_inference_log_profile_metrics` and `churn_inference_log_drift_metrics` tables

> **Note:** Lakehouse Monitoring requires the baseline table schema to be compatible with
> the inference log. The raw `telco_churn` table stores `TotalCharges` as STRING (11 rows
> have empty values in the IBM dataset). We create a view that casts it to DOUBLE to match
> the inference log schema.

In [0]:
import mlflow
from mlflow import MlflowClient

# Use Unity Catalog registry
mlflow.set_registry_uri("databricks-uc")
uc_client = MlflowClient()

model_name = f"{catalog}.{schema}.churn_classifier"
baseline_view = f"{catalog}.{schema}.churn_monitor_baseline"

# Derive model_id dynamically from the @champion alias
try:
    mv = uc_client.get_model_version_by_alias(model_name, "champion")
    model_id = f"{model_name}_v{mv.version}"
except Exception:
    model_id = f"{model_name}_v1"  # fallback if alias not yet set

print(f"Using model_id: {model_id}")

# Lakehouse Monitoring requires the baseline to have compatible column types.
# telco_churn.TotalCharges is STRING (IBM CSV has 11 empty values);
# churn_inference_log.TotalCharges is DOUBLE. We create a type-aligned baseline view.
spark.sql(f"""
    CREATE OR REPLACE VIEW {baseline_view} AS
    SELECT
        customerID,
        gender,
        SeniorCitizen,
        Partner,
        Dependents,
        TRY_CAST(tenure AS BIGINT),
        PhoneService,
        MultipleLines,
        InternetService,
        OnlineSecurity,
        OnlineBackup,
        DeviceProtection,
        TechSupport,
        StreamingTV,
        StreamingMovies,
        Contract,
        PaperlessBilling,
        PaymentMethod,
        MonthlyCharges,
        TRY_CAST(TotalCharges AS DOUBLE) AS TotalCharges,
        Churn,
        '{model_id}' AS model_id,
        now()        AS inference_ts,
        Churn        AS prediction
    FROM {catalog}.`00_shared`.telco_churn
    WHERE TotalCharges IS NOT NULL
""")

print(f"✓ Baseline view created : {baseline_view}")
print(f"  Row count             : {spark.table(baseline_view).count()}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import (
    MonitorInferenceLog,
    MonitorInferenceLogProblemType,
)

w = WorkspaceClient()
current_user = spark.sql("SELECT current_user()").first()[0]

full_table_name = inference_table  # set in cell above: catalog.schema.churn_inference_log
assets_dir      = f"/Users/{current_user}/monitoring/{schema}"

print(f"Creating monitor on : {full_table_name}")
print(f"Assets directory    : {assets_dir}")

try:
    monitor = w.quality_monitors.create(
        table_name=full_table_name,
        assets_dir=assets_dir,
        output_schema_name=f"{catalog}.{schema}",
        inference_log=MonitorInferenceLog(
            problem_type=MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION,
            model_id_col="model_id",
            prediction_col="prediction",
            label_col="Churn",
            timestamp_col="inference_ts",
            granularities=["1 day"],
        ),
        baseline_table_name=baseline_view,
    )
    print("✓ Monitor created!")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Monitor already exists — running a refresh instead.")
    else:
        raise

TODO:  I don't think this is necessary.  A refresh should be started upon creation.

In [0]:
# Trigger an immediate refresh to compute metrics
print("Triggering monitoring refresh (takes ~2-3 minutes)...")
refresh = w.quality_monitors.run_refresh(table_name=full_table_name)
print(f"Refresh ID: {refresh.refresh_id}")
print("Watch progress in the Catalog Explorer → churn_inference_log → Quality tab")

## Navigate to the Monitor Dashboard

1. Open **Catalog** in the left sidebar
2. Navigate: `workshop` → `<your_schema>` → `churn_inference_log`
3. Click the **Quality** tab
4. Click **View Dashboard**

TODO: OOTB Dashboard has errors that will need to be corrected manually.

Observe:
- Distribution comparison: baseline vs current window
- PSI (Population Stability Index) for each feature
- `tenure` should show the highest PSI (~0.28) — this is the injected drift

In [0]:
# Query the drift metrics table directly
# (Available after the refresh completes)
drift_table = f"`{catalog}`.`{schema}`.churn_inference_log_drift_metrics"

try:
    drift_df = spark.table(drift_table)
    print(f"Drift metrics table has {drift_df.count()} rows")
    display(
        drift_df
        .orderBy("drift_type", "column_name")
        .limit(20)
    )
except Exception:
    print("Drift metrics table not yet available — wait for the refresh to complete.")
    print("Re-run this cell in 2-3 minutes.")

## AI Gateway: Endpoint Usage Tracking

AI Gateway logs every request to `system.serving.endpoint_usage` — a system table that
gives you a complete picture of who is calling your endpoint and at what cost.

This data is available with approximately **1-hour latency** and is useful for:
- Auditing: who called the endpoint and when
- Cost attribution: token consumption per user or team
- Capacity planning: peak request rates and latency trends

In [0]:
# Query AI Gateway usage metrics from system tables
# Note: data appears with ~1 hour latency after requests are made
endpoint_name = f"churn_{safe_username}"

try:
    usage_df = spark.sql(f"""
        SELECT
            DATE(request_time)                 AS date,
            e.endpoint_name                    AS endpoint_name,
            COUNT(*)                           AS total_requests,
            SUM(input_token_count)             AS total_input_tokens,
            SUM(output_token_count)            AS total_output_tokens
        FROM system.serving.endpoint_usage u
        JOIN system.serving.served_entities e
            ON u.served_entity_id = e.served_entity_id
        WHERE requester = '{username}'
          AND request_time >= current_date() - INTERVAL 7 DAYS
        GROUP BY 1, 2
        ORDER BY 1 DESC
    """)
    print(f"Usage summary for: {endpoint_name}")
    display(usage_df)
except Exception as e:
    print(f"Note: Usage data not yet available (data has ~1hr latency).")
    print(f"Once requests flow through the endpoint, re-run this cell.")
    print(f"Details: {e}")

➡️ Next: `04_drift_alerts.ipynb` — create an automated alert that fires when drift is detected